# NEologd 辞書で新語・固有名詞をまとまりとして解析する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/04_japanese_tokenization/mecab_neologd.ipynb)

NLP クックブック 04-10 ／ 日本語の形態素解析と文分割

人名・作品名・組織名や新語が IPA 辞書では細かく切れてしまうので、1 語として扱いたい。ウェブ上の語彙から新語・固有表現を IPA 辞書に追加した MeCab 用のシステム辞書 mecab-ipadic-NEologd を使う。

In [1]:
# Colab で 5 分ほどかかる (辞書のダウンロードとビルド。できる辞書は 900MB ほど)
!apt-get install -y -q mecab libmecab-dev mecab-ipadic-utf8 > /dev/null
!git clone -q --depth 1 https://github.com/neologd/mecab-ipadic-neologd
!cd mecab-ipadic-neologd && ./bin/install-mecab-ipadic-neologd -n -y -u > ../neologd_install.log 2>&1; tail -n 3 ../neologd_install.log
!pip install -q fugashi
# インストール先は mecab-config --dicdir の下の mecab-ipadic-neologd
!ls -lh $(mecab-config --dicdir)/mecab-ipadic-neologd/sys.dic
!echo "米津玄師の新曲" | mecab -d $(mecab-config --dicdir)/mecab-ipadic-neologd

In [2]:
import subprocess
from fugashi import GenericTagger

dicdir = subprocess.run(["mecab-config", "--dicdir"], capture_output=True, text=True).stdout.strip()
NEOLOGD = f"{dicdir}/mecab-ipadic-neologd"
# fugashi は既定で /usr/local/etc/mecabrc を探すので、apt の MeCab の設定ファイルを -r で渡す
ipa = GenericTagger("-r /etc/mecabrc")                  # apt の IPA 辞書 (mecabrc の既定の辞書)
neo = GenericTagger(f"-r /etc/mecabrc -d {NEOLOGD}")    # NEologd

sents = [
    "米津玄師の新曲が鬼滅の刃の主題歌に決まった。",
    "大谷翔平と藤井聡太が話題になった。",
    "きゃりーぱみゅぱみゅがNHK紅白歌合戦に出場した。",
    "君の名は。を観てからインスタ映えする場所を巡った。",
    "逃げるは恥だが役に立つの恋ダンスが流行った。",
    "ほぼ日刊イトイ新聞とメルカリとZOZOTOWNを比べる。",
]
for s in sents:
    print("IPA    :", " | ".join(w.surface for w in ipa(s)))
    print("NEologd:", " | ".join(w.surface for w in neo(s)))
    print()

IPA    : 米津 | 玄 | 師 | の | 新曲 | が | 鬼 | 滅 | の | 刃 | の | 主題歌 | に | 決まっ | た | 。
NEologd: 米津玄師 | の | 新曲 | が | 鬼滅の刃 | の | 主題歌 | に | 決まっ | た | 。

IPA    : 大谷 | 翔 | 平 | と | 藤井 | 聡 | 太 | が | 話題 | に | なっ | た | 。
NEologd: 大谷翔平 | と | 藤井聡太 | が | 話題 | に | なっ | た | 。

IPA    : きゃ | り | ー | ぱみゅぱみゅが | NHK | 紅白 | 歌合戦 | に | 出場 | し | た | 。
NEologd: きゃりーぱみゅぱみゅ | が | NHK紅白歌合戦 | に | 出場 | し | た | 。

IPA    : 君 | の | 名 | は | 。 | を | 観 | て | から | インスタ | 映え | する | 場所 | を | 巡っ | た | 。
NEologd: 君の名は。 | を | 観 | て | から | インスタ映え | する | 場所 | を | 巡っ | た | 。

IPA    : 逃げる | は | 恥 | だ | が | 役に立つ | の | 恋 | ダンス | が | 流行っ | た | 。
NEologd: 逃げるは恥だが役に立つ | の | 恋ダンス | が | 流行っ | た | 。

IPA    : ほぼ | 日刊 | イトイ | 新聞 | と | メルカリ | と | ZOZOTOWN | を | 比べる | 。
NEologd: ほぼ日刊イトイ新聞 | と | メルカリ | と | ZOZOTOWN | を | 比べる | 。



In [3]:
# NEologd で固有名詞になった語の品詞 (細分類) と読み。素性は IPA 辞書と同じ並び
for s in sents:
    for w in neo(s):
        f = w.feature
        if f[1] == "固有名詞":
            print(w.surface, ",".join(f[:4]), f[7] if len(f) > 7 else "(読みなし: 未知語)", sep="\t")

米津玄師	名詞,固有名詞,人名,一般	ヨネヅケンシ
鬼滅の刃	名詞,固有名詞,一般,*	キメツノヤイバ
大谷翔平	名詞,固有名詞,人名,一般	オオタニショウヘイ
藤井聡太	名詞,固有名詞,人名,一般	フジイソウタ
きゃりーぱみゅぱみゅ	名詞,固有名詞,人名,一般	キャリーパミュパミュ
NHK紅白歌合戦	名詞,固有名詞,一般,*	エヌエイチケイコウハクウタガッセン
君の名は。	名詞,固有名詞,一般,*	キミノナハ
インスタ映え	名詞,固有名詞,一般,*	インスタハエ
逃げるは恥だが役に立つ	名詞,固有名詞,一般,*	ニゲルハハジダガヤクニタツ
恋ダンス	名詞,固有名詞,一般,*	コイダンス
ほぼ日刊イトイ新聞	名詞,固有名詞,人名,一般	ホボニッカンイトイシンブン
メルカリ	名詞,固有名詞,組織,*	メルカリ
ZOZOTOWN	名詞,固有名詞,一般,*	ゾゾタウン


In [4]:
# 収録されていない新しい語 (2020 年 9 月より後に広まった語など) は、IPA 辞書と同じく切れるか未知語になる
for s in ["推しの子のアニメを観た。", "ChatGPTと生成AIの話をした。"]:
    for w in neo(s):
        f = w.feature
        print(w.surface, ",".join(f[:4]), f[7] if len(f) > 7 else "(読みなし: 未知語)", sep="\t")
    print()

推し	動詞,自立,*,*	オシ
の子	名詞,固有名詞,人名,一般	ノコ
の	助詞,連体化,*,*	ノ
アニメ	名詞,一般,*,*	アニメ
を	助詞,格助詞,一般,*	ヲ
観	動詞,自立,*,*	ミ
た	助動詞,*,*,*	タ
。	記号,句点,*,*	。

ChatGPT	名詞,固有名詞,組織,*	(読みなし: 未知語)
と	助詞,格助詞,一般,*	ト
生成	名詞,サ変接続,*,*	セイセイ
AI	名詞,固有名詞,一般,*	エーアイ
の	助詞,連体化,*,*	ノ
話	名詞,サ変接続,*,*	ハナシ
を	助詞,格助詞,一般,*	ヲ
し	動詞,自立,*,*	シ
た	助動詞,*,*,*	タ
。	記号,句点,*,*	。



In [5]:
# 長い固有名詞が 1 語になると、その一部の語で検索したときに見つからなくなる (再現率が下がる)
docs = [
    "東京大学大学院情報理工学系研究科の共同研究。",
    "東京大学の入試が行われた。",
    "新型コロナウイルス感染症の影響でテレワークが広がった。",
    "コロナ禍でオンライン授業が増えた。",
    "コロナの影響で休校になった。",
]
for name, tagger in [("IPA", ipa), ("NEologd", neo)]:
    index = [set(w.surface for w in tagger(d)) for d in docs]
    for q in ["東京大学", "コロナ"]:
        print(f"{name:8}{q}: {[i for i, toks in enumerate(index) if q in toks]}")
print(" | ".join(w.surface for w in neo(docs[0])))
print(" | ".join(w.surface for w in neo(docs[2])))

# 対策の一例: 検索用の索引には両方の辞書のトークンを入れる (README も IPA 辞書との併用を勧めている)
index = [set(w.surface for w in ipa(d)) | set(w.surface for w in neo(d)) for d in docs]
for q in ["東京大学", "コロナ", "新型コロナウイルス感染症"]:
    print(f"両方    {q}: {[i for i, toks in enumerate(index) if q in toks]}")

IPA     東京大学: [0, 1]
IPA     コロナ: [2, 3, 4]
NEologd 東京大学: [1]
NEologd コロナ: [4]
東京大学大学院情報理工学系研究科 | の | 共同研究 | 。
新型コロナウイルス感染症 | の | 影響 | で | テレワーク | が | 広がっ | た | 。
両方    東京大学: [0, 1]
両方    コロナ: [2, 3, 4]
両方    新型コロナウイルス感染症: [2]


- mecab-ipadic-NEologd は、はてなキーワード・郵便番号データ・駅名・ニュース記事・ウェブの文書などから集めた新語・固有表現を IPA 辞書 (mecab-ipadic-2.7.0-20070801) に追加したシステム辞書 (佐藤敏紀氏ら、Apache License 2.0)。IPA 辞書と同じ素性の並びで読みも付く。apt の mecab-ipadic-utf8 と pip の `ipadic` (前のレシピ「IPA 辞書で MeCab を使う」) は同じ版の IPA 辞書。NEologd は IPA 辞書の語の読みや生起コストにもパッチを当てるので、IPA 辞書との差がすべて新語によるものとは限らない。
- **更新は止まっている。** README には「少なくとも毎週 2 回更新」とあるが、master の最後のコミットは 2020 年 9 月 14 日 (シードのデータは 2020-09-10 版) で、それ以降の語は入っていない (2026 年 11 月時点で、リポジトリはアーカイブされていない)。上の例では、収録されていない「推しの子」が「推し / の子」に切れて「の子」が人名になり、「ChatGPT」は読みのない未知語になる。
- 固有名詞の細分類は粗く、誤りもある (上の例では「ほぼ日刊イトイ新聞」が「人名」)。README の「欠点」にも、分類が不十分なこと、固有表現ではない語も固有表現として登録されていること、読みの誤りがあることが書かれている。入力は、NEologd の正規化ルールを実装した neologdn (3 章「文字列処理と正規化」の「neologdn で日本語の表記ゆれを正規化する」) で正規化してから解析するとよい。
- 長い固有名詞が 1 語になるので、トークン単位で「東京大学」「コロナ」と検索しても「東京大学大学院情報理工学系研究科」「新型コロナウイルス感染症」は見つからない。README も、ウェブの文書の解析では標準の IPA 辞書との併用を勧めている。単語の単位を評価データや学習済みモデルに合わせる必要がある場合は使わない。
- インストーラーは元の IPA 辞書の tar.gz を自分でダウンロードし (SHA-1 で照合)、パッチを当ててビルドする。最初のダウンロード先 osdn.net は検証時 (2026 年 11 月) 名前解決できず、SourceForge から取得された。`-n` は GitHub の最新版に更新してからインストール、`-y` はテスト結果の確認を省略、`-u` はインストール先が自分のディレクトリでなくても sudo を使わずに `make install` する (付けないと Colab では `sudo make install` を通る)。インストール先は `mecab-config --dicdir` の下 (Colab では `/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ipadic-neologd`) で、`-p ディレクトリ` で変えられる。
- README の目安では、標準のインストールに空きメモリが最低 2GB (推奨 6GB) 必要。手元の検証では、ビルド中の最大のプロセスのメモリ使用量 (最大常駐サイズ) は約 1.6GB、できた `sys.dic` は `ls -lh` で 866M (README では約 910MB) だった。`--ignore_adverb` などの `--ignore_*` で一部の辞書を外すと小さくなり (README によれば全部指定して約 300MB 減る)、`-a` では標準では入らない低頻度の日時・数量表現などもすべて入る。Colab のランタイムを作り直すとインストールはやり直しになる。